# Item-to-Item Recommendation System Using BoW
Bag-of-Words computes the token counts and we can expect a similar performance to TF-IDF, with slightly different performance in specific scenarios. 

In [1]:
from pathlib import Path
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import linear_kernel

In [2]:
wines_features = pd.read_csv(Path("../data/processed/wines_features.csv"))
wines_features = wines_features[wines_features["content_text"].notna()].copy()
wines_features = wines_features[wines_features["content_text"].str.strip() != ""].copy()
wines_features = wines_features.reset_index(drop=True)

print(wines_features.shape)
wines_features.head()

(100646, 36)


,WineID,WineName,Type,Elaborate,Grapes,Harmonize,ABV,Body,Acidity,Country,...,Acidity_nat,Country_nat,RegionName_nat,WineryName_nat,Grapes_nat,Harmonize_nat,ABV_bin,ABV_nat,content_text,bert_text
0,100001,Espumante Moscatel,Sparkling,Varietal/100%,['Muscat/Moscato'],"['Pork', 'Rich Fish', 'Shellfish']",7.5,Medium-bodied,High,Brazil,...,high,brazil,serra gaúcha,casa perini,['muscat moscato'],"['pork', 'rich fish', 'shellfish']",abv_low,7.5% abv,type_sparkling elaborate_varietal_100 body_med...,espumante moscatel. This wine is a medium bodi...
1,100002,Ancellotta,Red,Varietal/100%,['Ancellotta'],"['Beef', 'Barbecue', 'Codfish', 'Pasta', 'Pizz...",12.0,Medium-bodied,Medium,Brazil,...,medium,brazil,serra gaúcha,casa perini,['ancellotta'],"['beef', 'barbecue', 'codfish', 'pasta', 'pizz...",abv_12_13,12.0% abv,type_red elaborate_varietal_100 body_medium_bo...,"ancellotta. This wine is a medium bodied, medi..."
2,100003,Cabernet Sauvignon,Red,Varietal/100%,['Cabernet Sauvignon'],"['Beef', 'Lamb', 'Poultry']",12.0,Full-bodied,High,Brazil,...,high,brazil,serra gaúcha,castellamare,['cabernet sauvignon'],"['beef', 'lamb', 'poultry']",abv_12_13,12.0% abv,type_red elaborate_varietal_100 body_full_bodi...,cabernet sauvignon. This wine is a full bodied...
3,100004,Virtus Moscato,White,Varietal/100%,['Muscat/Moscato'],['Sweet Dessert'],12.0,Medium-bodied,Medium,Brazil,...,medium,brazil,serra gaúcha,monte paschoal,['muscat moscato'],['sweet dessert'],abv_12_13,12.0% abv,type_white elaborate_varietal_100 body_medium_...,"virtus moscato. This wine is a medium bodied, ..."
4,100005,Maison de Ville Cabernet-Merlot,Red,Assemblage/Bordeaux Red Blend,"['Cabernet Sauvignon', 'Merlot']","['Beef', 'Lamb', 'Game Meat', 'Poultry']",11.0,Full-bodied,Medium,Brazil,...,medium,brazil,serra gaúcha,aurora,"['cabernet sauvignon', 'merlot']","['beef', 'lamb', 'game meat', 'poultry']",abv_11_12,11.0% abv,type_red elaborate_assemblage_bordeaux_red_ble...,maison de ville cabernet merlot. This wine is ...


In [3]:
bow = CountVectorizer()
bow_matrix = bow.fit_transform(wines_features["content_text"])

print(bow_matrix.shape)

(100646, 93506)


In [4]:
wineid_to_idx = pd.Series(wines_features.index, index=wines_features["WineID"]).drop_duplicates()

In [5]:
def recommend_similar_wines_bow(wine_id, top_k=5):
    if wine_id not in wineid_to_idx:
        raise ValueError(f"WineID {wine_id} not found.")
    
    idx = wineid_to_idx[wine_id]
    
    sim_scores = linear_kernel(bow_matrix[idx:idx+1], bow_matrix).flatten()
    top_indices = sim_scores.argsort()[::-1][1:top_k+1]
    
    results = wines_features.loc[top_indices, [
        "WineID", "WineName", "Type", "Country", "RegionName",
        "Body", "Acidity", "Grapes", "Harmonize"
    ]].copy()
    
    results["similarity_score"] = sim_scores[top_indices]
    results["similarity_score"] = results["similarity_score"].round(3)
    
    return results.reset_index(drop=True)

In [6]:
sample_wine_id = wines_features.iloc[0]["WineID"]
recommend_similar_wines_bow(sample_wine_id, top_k=5)

,WineID,WineName,Type,Country,RegionName,Body,Acidity,Grapes,Harmonize,similarity_score
0,100073,Moscatel,Sparkling,Brazil,Serra Gaúcha,Medium-bodied,High,['Muscat/Moscato'],"['Pork', 'Rich Fish', 'Shellfish']",11.0
1,101048,Moscatel Espumante,Sparkling,Brazil,Serra Gaúcha,Medium-bodied,High,['Muscat/Moscato'],"['Pork', 'Rich Fish', 'Shellfish']",11.0
2,101459,Lendas do Brasil A Moscatel Espumante,Sparkling,Brazil,Serra Gaúcha,Medium-bodied,High,['Muscat/Moscato'],"['Pork', 'Rich Fish', 'Shellfish']",11.0
3,100698,San Martin Moscatel Rosé Brut,Sparkling,Brazil,Serra Gaúcha,Medium-bodied,High,['Muscat/Moscato'],"['Pork', 'Rich Fish', 'Shellfish']",11.0
4,101281,Moscatel Rosé,Sparkling,Brazil,Serra Gaúcha,Medium-bodied,High,['Muscat/Moscato'],"['Pork', 'Rich Fish', 'Shellfish']",11.0


Interestingly, compared to TF-IDF recsys that recommended one white wine and wines of various acidity and harmonize values, the BoW outperformed it significantly by giving an exact match in those columns.

In [9]:
sample_ids = wines_features["WineID"].sample(3, random_state=42).tolist()

for wine_id in sample_ids:
    print(f"\nRecommendations for WineID {wine_id}")
    display(recommend_similar_wines_bow(wine_id, top_k=5))


Recommendations for WineID 121539


,WineID,WineName,Type,Country,RegionName,Body,Acidity,Grapes,Harmonize,similarity_score
0,118143,Les Hauts du Village Rasteau,Red,France,Rasteau,Full-bodied,High,"['Syrah/Shiraz', 'Grenache', 'Mourvedre']","['Beef', 'Lamb', 'Game Meat']",13.0
1,132642,Les Grandes Terres Rasteau,Red,France,Rasteau,Full-bodied,High,"['Mourvedre', 'Syrah/Shiraz', 'Grenache']","['Beef', 'Lamb', 'Game Meat']",13.0
2,130079,La Réserve Rasteau,Red,France,Rasteau,Full-bodied,High,"['Syrah/Shiraz', 'Grenache', 'Mourvedre']","['Beef', 'Lamb', 'Game Meat']",13.0
3,135184,Rasteau,Red,France,Rasteau,Full-bodied,High,"['Syrah/Shiraz', 'Grenache', 'Mourvedre']","['Beef', 'Lamb', 'Game Meat']",13.0
4,135084,Les Héritiers Rasteau,Red,France,Rasteau,Full-bodied,High,"['Syrah/Shiraz', 'Grenache', 'Mourvedre']","['Beef', 'Lamb', 'Game Meat']",13.0



Recommendations for WineID 169173


,WineID,WineName,Type,Country,RegionName,Body,Acidity,Grapes,Harmonize,similarity_score
0,169314,Don Nicanor Chardonnay-Viognier,White,Argentina,Mendoza,Medium-bodied,Medium,"['Chardonnay', 'Viognier']","['Pork', 'Game Meat', 'Rich Fish', 'Shellfish'...",17.0
1,169642,Nuna Estate White Blend,White,Argentina,Mendoza,Medium-bodied,Medium,"['Viognier', 'Chardonnay', 'Sauvignon Blanc']","['Pork', 'Game Meat', 'Rich Fish', 'Shellfish'...",17.0
2,169809,Santa Isabel Chardonnay-Viognier,White,Argentina,Mendoza,Medium-bodied,Low,"['Chardonnay', 'Viognier']","['Pork', 'Game Meat', 'Rich Fish', 'Shellfish'...",17.0
3,169121,V Blend,White,Argentina,Mendoza,Full-bodied,Medium,"['Chardonnay', 'Gewürztraminer', 'Viognier']","['Pork', 'Game Meat', 'Rich Fish', 'Shellfish'...",16.0
4,168294,Serie A Chardonnay-Viognier,White,Argentina,Mendoza,Medium-bodied,Low,"['Chardonnay', 'Viognier']","['Pork', 'Game Meat', 'Rich Fish', 'Shellfish'...",16.0



Recommendations for WineID 198191


,WineID,WineName,Type,Country,RegionName,Body,Acidity,Grapes,Harmonize,similarity_score
0,200091,Chardonnay,White,Israel,Galilee,Medium-bodied,Medium,['Chardonnay'],"['Pork', 'Rich Fish', 'Vegetarian', 'Poultry']",11.0
1,198051,Chardonnay,White,Israel,Negev,Medium-bodied,Medium,['Chardonnay'],"['Pork', 'Rich Fish', 'Vegetarian', 'Poultry']",11.0
2,196055,Unoaked Chardonnay,White,Israel,Galilee,Medium-bodied,Medium,['Chardonnay'],"['Pork', 'Rich Fish', 'Vegetarian', 'Poultry']",11.0
3,198997,Rhyton Chardonnay,White,Israel,Galilee,Medium-bodied,Medium,['Chardonnay'],"['Pork', 'Rich Fish', 'Vegetarian', 'Poultry']",11.0
4,198371,Chardonnay,White,Israel,Galilee,Medium-bodied,Medium,['Chardonnay'],"['Pork', 'Rich Fish', 'Vegetarian', 'Poultry']",11.0


Same goes for other wines we tested out, giving an almost exact match, indicating that the BoW model outperforms the TF-IDF model.

In [7]:
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.preprocessing import Normalizer
import plotly.express as px

In [8]:
plot_sample = wines_features.sample(3000, random_state=42).reset_index(drop=True)
plot_bow = bow.transform(plot_sample["content_text"])

svd = TruncatedSVD(n_components=50, random_state=42)
plot_lsa = svd.fit_transform(plot_bow)
plot_lsa = Normalizer(copy=False).fit_transform(plot_lsa)

pca = PCA(n_components=2, random_state=42)
plot_2d = pca.fit_transform(plot_lsa)

plot_df = plot_sample[["WineID", "WineName", "Type", "Country", "RegionName"]].copy()
plot_df["pc1"] = plot_2d[:, 0]
plot_df["pc2"] = plot_2d[:, 1]

fig = px.scatter(
    plot_df,
    x="pc1",
    y="pc2",
    color="Type",
    hover_data=["WineID", "WineName", "Country", "RegionName"],
    title="2D projection of wine representations from Bag of Words features"
)

fig.show()

Similarly to the TF-IDF model there is a clear distinction between the red and white wines on the plot, while the others are mixed in the middle. This serves as a good visual check in terms of the wine vectors.